In [1]:
# Import pandas and load the raw medical insurance CSV dataset into a DataFrame
import pandas as pd

df = pd.read_csv("C:\\Users\\Radwa\\Downloads\\archive (1)\\medical_insurance.csv")

df.head()

,person_id,age,sex,region,urban_rural,income,education,marital_status,employment_status,household_size,...,liver_disease,arthritis,mental_health,proc_imaging_count,proc_surgery_count,proc_physio_count,proc_consult_count,proc_lab_count,is_high_risk,had_major_procedure
0,75722,52,Female,North,Suburban,22700.0,Doctorate,Married,Retired,3,...,0,1,0,1,0,2,0,1,0,0
1,80185,79,Female,North,Urban,12800.0,No HS,Married,Employed,3,...,0,1,1,0,0,1,0,1,1,0
2,19865,68,Male,North,Rural,40700.0,HS,Married,Retired,5,...,0,0,1,1,0,2,1,0,1,0
3,76700,15,Male,North,Suburban,15600.0,Some College,Married,Self-employed,5,...,0,0,0,1,0,0,1,0,0,0
4,92992,53,Male,Central,Suburban,89600.0,Doctorate,Married,Self-employed,2,...,0,1,0,2,0,1,1,0,1,0


In [2]:
# Fill missing values in 'alcohol_freq' column using the most frequent value (mode)

mode_val = df['alcohol_freq'].mode()[0]
df['alcohol_freq'] = df['alcohol_freq'].fillna(mode_val)

print(f"Filled with: '{mode_val}'")
print(f"Remaining nulls: {df['alcohol_freq'].isna().sum()}")

Filled with: 'Occasional'
Remaining nulls: 0


In [3]:
# Flag suspicious rows: people aged 95+ who show signs of being data entry errors
# (e.g., still employed, few medications, no chronic diseases, or active smokers)

suspicious_mask = (
    (df['age'] >= 95) &
    (
        df['employment_status'].isin(['Employed', 'Self-employed']) |
        (df['medication_count'] < 3) |
        (df[['hypertension','diabetes','asthma','copd',
             'cardiovascular_disease','cancer_history',
             'kidney_disease','liver_disease',
             'arthritis','mental_health']].sum(axis=1) == 0) |
        (df['smoker'] == 'Current')
    )
)

MIN_AGE = 18  
df['is_suspicious'] = suspicious_mask
print(f"Suspicious rows: {suspicious_mask.sum()}")

Suspicious rows: 159


In [4]:
# On the clean (non-suspicious) subset, compute median age grouped by 5 demographic columns
# and also by 4 columns as a fallback — these will be used to impute suspicious ages

clean = df[~df['is_suspicious']].copy()

group5 = ['education', 'employment_status', 'marital_status', 'alcohol_freq', 'smoker']
med5 = clean.groupby(group5)['age'].median().reset_index()
med5.columns = group5 + ['median_5']

group4 = ['education', 'employment_status', 'marital_status', 'alcohol_freq']
med4 = clean.groupby(group4)['age'].median().reset_index()
med4.columns = group4 + ['median_4']

global_med = clean['age'].median()
print(f"Global median age: {global_med}")

Global median age: 48.0


In [5]:
# For each suspicious row, look up the most specific contextual median age available
# (5-group match → 4-group match → global median) and store as the replacement age
suspicious_df = df[df['is_suspicious']].copy()

suspicious_df = suspicious_df.merge(med5, on=group5, how='left')

suspicious_df = suspicious_df.merge(med4, on=group4, how='left')

suspicious_df['contextual_median'] = (
    suspicious_df['median_5']
    .combine_first(suspicious_df['median_4'])
    .combine_first(pd.Series(global_med, index=suspicious_df.index))
)

suspicious_df['new_age'] = suspicious_df['contextual_median'].clip(lower=MIN_AGE).round()

print(suspicious_df[['person_id','age','contextual_median','new_age']].head(10))

   person_id  age  contextual_median  new_age
0      33076   96               47.0     47.0
1      38358   95               48.0     48.0
2      32613  100               44.0     44.0
3      78889  100               45.0     45.0
4      68362   95               48.0     48.0
5      23356   98               48.0     48.0
6      43600   99               48.0     48.0
7      87531   98               48.0     48.0
8       3717   99               47.0     47.0
9      17437   97               47.0     47.0


In [6]:
# Apply the computed replacement ages back to the original DataFrame and remove the flag column

df.loc[df['is_suspicious'], 'age'] = (
    suspicious_df.set_index('person_id')['new_age']
    .reindex(df.loc[df['is_suspicious'], 'person_id'].values)
    .values
)
df.drop(columns=['is_suspicious'], inplace=True)

print("Done!")
print(df['age'].describe())

Done!
count    100000.000000
mean         47.441850
std          15.862763
min           0.000000
25%          37.000000
50%          48.000000
75%          58.000000
max         100.000000
Name: age, dtype: float64


In [7]:
# Print final age statistics to verify the imputation worked correctly
print(f"Min age:    {df['age'].min()}")
print(f"Max age:    {df['age'].max()}")
print(f"Avg age:    {df['age'].mean():.2f}")
print(f"Total rows: {len(df)}")
print(f"Remaining suspicious: {((df['age'] >= 95) & suspicious_mask.reindex(df.index, fill_value=False)).sum()}")


Min age:    0
Max age:    100
Avg age:    47.44
Total rows: 100000
Remaining suspicious: 0


In [8]:
# Flag rows where diabetes = 1 but HbA1c < 6.5 (medically inconsistent — not actually diabetic)

flag = (df['diabetes'] == 1) & (df['hba1c'] < 6.5)
df['diabetes_hba1c_flag'] = flag.astype(int)

print(f"Flagged rows: {flag.sum()}")
print(df['diabetes_hba1c_flag'].value_counts())

Flagged rows: 1162
diabetes_hba1c_flag
0    98838
1     1162
Name: count, dtype: int64


In [10]:
# Correct the inconsistency: set diabetes = 0 for flagged rows where HbA1c doesn't support the diagnosis
df.loc[(df['diabetes'] == 1) & (df['hba1c'] < 6.5), 'diabetes'] = 0

In [11]:
# Verify the fix: confirm no remaining diabetes/HbA1c mismatches exist
flag = (df['diabetes'] == 1) & (df['hba1c'] < 6.5)

print(f"Remaining flagged rows: {flag.sum()}")

Remaining flagged rows: 0


In [12]:
# Flag rows where 'had_major_procedure' = 1 but surgery count = 0 (contradictory data)

mismatch = (df['had_major_procedure'] == 1) & (df['proc_surgery_count'] == 0)
df['procedure_mismatch_flag'] = mismatch.astype(int)

print(f"Flagged rows: {mismatch.sum()}")
print(df['procedure_mismatch_flag'].value_counts())

Flagged rows: 4451
procedure_mismatch_flag
0    95549
1     4451
Name: count, dtype: int64


In [13]:
# Correct the mismatch: set had_major_procedure = 0 where no surgery was recorded
df.loc[(df['had_major_procedure'] == 1) & (df['proc_surgery_count'] == 0), 'had_major_procedure'] = 0

In [14]:
# Verify the procedure mismatch fix — should now return 0 remaining conflicts
mismatch = (df['had_major_procedure'] == 1) & (df['proc_surgery_count'] == 0)
print(mismatch.sum())

0


In [15]:
# Cap extreme outliers in financial columns at the 1st and 99th percentiles (Winsorization)
financial_cols = [
    'annual_medical_cost', 'annual_premium', 'monthly_premium',
    'avg_claim_amount', 'total_claims_paid', 'income'
]

for col in financial_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f"{col}: clipped to [{lower:.0f}, {upper:.0f}]")

annual_medical_cost: clipped to [290, 15294]
annual_premium: clipped to [244, 2142]
monthly_premium: clipped to [20, 179]
avg_claim_amount: clipped to [0, 4938]
total_claims_paid: clipped to [0, 10565]
income: clipped to [5600, 232200]


In [16]:
# Display descriptive statistics for financial columns after clipping to verify the result
df[financial_cols].describe()

,annual_medical_cost,annual_premium,monthly_premium,avg_claim_amount,total_claims_paid,income
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.00000
mean,2949.692555,574.211946,47.850945,634.427688,1329.124998,49077.29900
std,2723.808422,343.176346,28.597896,916.457835,1952.475357,41786.28959
min,290.428400,243.860000,20.320000,0.000000,0.000000,5600.00000
25%,1175.117500,352.070000,29.340000,0.000000,0.000000,21100.00000
50%,2082.575000,463.585000,38.630000,318.015000,642.545000,36200.00000
75%,3707.957500,666.697500,55.560000,872.215000,1795.522500,62200.00000
max,15293.682200,2142.050100,178.500900,4937.528800,10564.555500,232200.00000


In [18]:
# Print the final shape (rows × columns) of the cleaned dataset
print(f"Final shape: {df.shape}")

Final shape: (100000, 56)


In [19]:
# Confirm there are no remaining null values anywhere in the DataFrame
print(f"Remaining nulls: {df.isnull().sum().sum()}")

Remaining nulls: 0


In [20]:
# Check for any fully duplicate rows in the cleaned dataset
print(f"Duplicate rows: {df.duplicated().sum()}")

Duplicate rows: 0


In [21]:
# Export the fully cleaned dataset to a CSV file for downstream use
df.to_csv('medical_insurance_cleaned.csv', index=False)